## 1. Environment Setup and Library Imports

In [ ]:
import os
import sys
import random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset

# -------------------------------------------------------------------------
# Path resolution to access root directory and 'src' modules
# -------------------------------------------------------------------------
PROJECT_ROOT = Path.cwd().parent if "__file__" not in globals() else Path(__file__).resolve().parent.parent
DATA_DIR = PROJECT_ROOT / "data"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import local custom modules
from src.pkldataset import PKLDataset, class_to_onehot
from src.helpers import set_seed, get_model, train_model, eval_model

# Hardware configuration and loss function
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.CrossEntropyLoss()

# Random seeds (10 seeds for statistical significance)
seeds = [101, 202, 303, 404, 505, 606, 707, 808, 909, 1001]

# Base sheet thickness used for HC training subsets
source_thickness = "T197"

print(f"Device: {device}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"HC Source thickness: {source_thickness}")

## 2. Configuration (RP Force Sensor)

In [ ]:
import sys
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset

# -------------------------------------------------------------------------
# Path resolution
# -------------------------------------------------------------------------
PROJECT_ROOT = Path.cwd().parent if "__file__" not in globals() else Path(__file__).resolve().parent.parent
DATA_DIR = PROJECT_ROOT / "data"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.pkldataset import PKLDataset, class_to_onehot

# -------------------------------------------------------------------------
# Stratified Sampling Helper for HC Subsets
# -------------------------------------------------------------------------
def get_balanced_transfer_subset(dataset, total_samples, seed=42):
    """
    Extracts an exactly class-balanced subset from a PKLDataset or Subset instance.
    For example: 20 samples -> 2 samples per wear class (R05..R50).
    """
    np.random.seed(seed)
    num_classes = 10
    samples_per_class = total_samples // num_classes
    
    if isinstance(dataset, Subset):
        base_dataset = dataset.dataset
        available_indices = dataset.indices
    else:
        base_dataset = dataset
        available_indices = list(range(len(dataset.files)))
    
    class_indices = {i: [] for i in range(num_classes)}
    for idx in available_indices:
        filename = base_dataset.files[idx]
        class_label = filename.split('_')[0]
        c_idx = int(np.argmax(class_to_onehot[class_label]))
        class_indices[c_idx].append(idx)
        
    selected_indices = []
    for c_idx, indices in class_indices.items():
        chosen = np.random.choice(indices, size=samples_per_class, replace=False)
        selected_indices.extend(chosen)
        
    return Subset(base_dataset, selected_indices)

# -------------------------------------------------------------------------
# Dataset Paths & Subsets (RP Force Sensor)
# -------------------------------------------------------------------------
# Source domain training path: HC only (T197 / RP)
train_path_hc = str(DATA_DIR / "HC" / "T197" / "RP")

# Sample sizes extracted from HC (T197)
sample_sizes = [20, 50, 100, 200, 300, 400, 500]
subset_names = [f"train_{size}" for size in sample_sizes]

# Evaluation paths: 100% of samples across all DC sheet thicknesses
val_paths = [
    str(DATA_DIR / "DC" / "T185" / "RP"),
    str(DATA_DIR / "DC" / "T188" / "RP"),
    str(DATA_DIR / "DC" / "T191" / "RP"),
    str(DATA_DIR / "DC" / "T194" / "RP"),
    str(DATA_DIR / "DC" / "T197" / "RP"),
]

# Random seeds for reproducibility across runs (10 seeds)
seeds = [101, 202, 303, 404, 505, 606, 707, 808, 909, 1001]

# Load full HC dataset to draw subsets from
full_hc_dataset = PKLDataset(folder_path=train_path_hc, modality="single")

# Results container: results[subset_name][val_path] -> list of accuracies
results = {s_name: {vp: [] for vp in val_paths} for s_name in subset_names}

## 3. Validation Preloading - RP

In [ ]:
# -------------------------------------------------------------------------
# Preload full DC validation datasets into RAM to avoid redundant disk I/O
# -------------------------------------------------------------------------
print("Preloading full DC validation datasets into RAM...")
cached_val_tensors = {}

for vp in val_paths:
    # Full dataset without splits: pure zero-shot generalization
    temp_ds = PKLDataset(folder_path=vp, modality="single")
    temp_loader = DataLoader(temp_ds, batch_size=256, shuffle=False)
    
    all_inputs, all_targets = [], []
    for x, y in temp_loader:
        all_inputs.append(x)
        all_targets.append(y)
        
    cached_val_tensors[vp] = {
        "inputs": torch.cat(all_inputs, dim=0),
        "targets": torch.cat(all_targets, dim=0)
    }

print("Validation datasets successfully preloaded into RAM.")
print(f"\nHC Source Training Path: {train_path_hc}")
print(f"DC Zero-Shot Validation Paths: {val_paths}")
print(f"Total Random Seeds: {len(seeds)}")

## 4. Training and Evaluation - RP

In [ ]:
# -------------------------------------------------------------------------
# Evaluation helper using in-memory cached tensors
# -------------------------------------------------------------------------
def fast_eval(model, inputs_tensor, targets_tensor, batch_size=256, device="cuda"):
    model.eval()
    correct = 0
    total = targets_tensor.size(0)
    with torch.no_grad():
        for i in range(0, total, batch_size):
            x = inputs_tensor[i:i + batch_size].to(device)
            y = targets_tensor[i:i + batch_size].to(device)
            preds = model(x).argmax(dim=1)
            targets_idx = y.argmax(dim=1)
            correct += (preds == targets_idx).sum().item()
    return 100.0 * correct / total

# -------------------------------------------------------------------------
# Scratch Training on HC Subsets (train_20 to train_500) -> Zero-Shot on DC
# -------------------------------------------------------------------------

for seed in seeds:
    print(f"\n==========================================")
    print(f">>> Running HC Subsets RP Experiment for Seed: {seed}")
    print(f"==========================================")
    set_seed(seed)

    for s_name, n_samples in zip(subset_names, sample_sizes):
        print(f"\n--- Training on HC Subset: {s_name} ({n_samples} samples) ---")

        # Extract class-balanced subset from the HC source dataset
        train_subset_hc = get_balanced_transfer_subset(
            dataset=full_hc_dataset,
            total_samples=n_samples,
            seed=seed
        )
        batch_sz = min(32, n_samples)
        train_loader_hc = DataLoader(train_subset_hc, batch_size=batch_sz, shuffle=True)

        # Initialize fresh model from scratch (no weights transferred, no fine-tuning)
        model = get_model(input_channels=1).to(device)
        optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
        scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=25, gamma=0.1)

        # Train model on the current HC subset
        model = train_model(
            model=model,
            loader=train_loader_hc,
            criterion=criterion,
            optimizer=optimizer,
            scheduler=scheduler,
            num_epochs=100,
            device=device
        )

        # Direct Zero-Shot evaluation across all DC sheet thicknesses from RAM
        for vp in val_paths:
            acc = fast_eval(
                model=model,
                inputs_tensor=cached_val_tensors[vp]["inputs"],
                targets_tensor=cached_val_tensors[vp]["targets"],
                device=device
            )
            results[s_name][vp].append(acc)
            thickness_name = vp.split("/")[-2] if "/" in vp else vp.split("\\")[-2]
            print(f"[Seed {seed}] {s_name} -> {thickness_name}: {acc:.2f}%")

## 5. Summary of Results - RP

In [ ]:
# -------------------------------------------------------------------------
# Compile evaluation summary across all DC sheet thicknesses (RP)
# -------------------------------------------------------------------------
summary_data = []

for s_name in subset_names:
    row = {"Subset": s_name}
    for vp in val_paths:
        accs = results[s_name][vp]
        mean_acc = np.mean(accs)
        std_acc = np.std(accs)
        col_name = vp.split("/")[-2] if "/" in vp else vp.split("\\")[-2]
        row[col_name] = f"{mean_acc:.2f} ± {std_acc:.2f}"
    summary_data.append(row)

df_rp = pd.DataFrame(summary_data)
df_rp.to_csv("wear_RP_all_thicknesses_summary.csv", index=False)

print("Consolidated RP results across HC subsets:")
display(df_rp)